# XCT Membrane Segmentation — U-Net (segmentation_models / Keras)

Trains a U-Net on whichever slices are marked `corrected` in the annotation
notebook's manifest, then:

- predicts on any sampled slices that are still unlabelled (section 8-9) —
  skipped automatically once every sample is corrected, and
- segments the **entire original stack** (section 10) — every slice, not
  just the 50 samples, using ground truth directly wherever it exists and
  the model everywhere else.

Reuses the same `output_dir` layout (`images/`, `masks_corrected/`,
`sample_manifest.csv`) as the annotation notebook — run this pointed at the
same output folder, and also set `CONFIG["stack_path"]` to the same original
stack file used there (needed for section 11).

**Why patches, not full images:** slices are ~2028x2028 — training on small
random crops (patches) instead of full images keeps memory/compute
manageable, and turns each labelled slice into many distinct training crops
per epoch via random position + augmentation, which matters a lot when
ground truth is limited. Inference stitches patch predictions back together
over a sliding window.

**Known compatibility gotcha:** `segmentation_models` hasn't been updated
for Keras 3 / TensorFlow >= 2.16. If you're on a recent TensorFlow and hit
import errors, either pin `tensorflow<2.16`, or install `tf-keras` and set
`TF_USE_LEGACY_KERAS=1` before importing TensorFlow.


## How to use this notebook

**Pipeline position:** notebook **2 of 3**. Run this after
`xct_membrane_segmentation.ipynb` has produced at least a handful of
hand-corrected slices — it trains a U-Net on those, then uses the trained
model to segment the rest of the stack.

**What it does, in order:**

1. Loads `sample_manifest.csv` and splits it into corrected (training) vs.
   not-yet-corrected slices.
2. Trains a U-Net on random augmented patches from the corrected slices.
3. Runs the trained model on the still-uncorrected sampled slices (sections
   8–9) so you can preview quality before committing to the full stack.
4. Segments the entire ~4000-slice stack (section 11), using your real
   hand-corrected masks wherever they exist and model predictions everywhere
   else.

**Before running, edit the `CONFIG` cell (section 1) below.** At minimum:

| Variable | What to set it to |
|---|---|
| `output_dir` | Same `output_dir` used in `xct_membrane_segmentation.ipynb` |
| `stack_path` | Same stack file used there (only needed for section 11) |
| `model_dir` | Where to save trained model checkpoints |
| `epochs` | Lower this for a quick test run before committing to a full training run |

The rest of `CONFIG` (backbone, patch size, batch size, learning rate) has
working defaults for a single-GPU workstation — revisit them if training is
too slow, runs out of memory, or the loss/IoU curves in section 7 look wrong.

**Tip:** if you already have a saved model and just want to re-run inference,
skip section 7 (Train) and run section 7b instead to reload it.

**Compatibility note:** `segmentation_models` doesn't support Keras 3 /
TensorFlow ≥ 2.16. Pin `tensorflow<2.16`, or install `tf-keras` and set
`TF_USE_LEGACY_KERAS=1` before importing TensorFlow, if you hit import errors.


## 0. Setup

In [ ]:
# pip install "tensorflow<2.16" segmentation-models albumentations scikit-learn tifffile tqdm

import os
os.environ["SM_FRAMEWORK"] = "tf.keras"  # must be set BEFORE importing segmentation_models

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from tqdm.auto import tqdm

import tensorflow as tf
import segmentation_models as sm
import albumentations as A
from sklearn.model_selection import train_test_split
from skimage.color import label2rgb

print("TensorFlow:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices("GPU"))


## 1. Config

In [ ]:
CONFIG = {
    # same output_dir as the annotation notebook — reuses images/, masks_corrected/, sample_manifest.csv
    "output_dir": Path("path/to/output"),
    "model_dir": Path("/path/to/model/output"),

    # same original stack used in xct_membrane_segmentation.ipynb — only needed for section 11
    # (full-stack segmentation); sections 1-9 only touch the 50 sampled slices via output_dir
    "stack_path": Path("/path/to/tiff/stack"),

    "phase_names": ["voidage", "membrane", "polymer_cartridge"],
    "n_classes": 3,

    # --- model ---
    "backbone": "resnet34",       # imagenet-pretrained encoder; needs 3-channel input
    "patch_size": 384,
    "learning_rate": 1e-4,

    # --- training ---
    "batch_size": 8,
    "patches_per_epoch": 320,     # random patches sampled per epoch (train_steps = this / batch_size)
    "val_tiles_per_image": 9,     # deterministic grid tiles per validation slice (fixed, not resampled each epoch)
    "val_fraction": 0.2,          # fraction of the corrected slices held out for validation
    "epochs": 250,
    "random_seed": 42,

    # --- inference ---
    "inference_stride_fraction": 0.5,  # sliding-window stride as a fraction of patch_size (0.5 = 50% overlap)
    "inference_batch_size": 32,        # separate from training batch_size — no gradients to store, so this
                                        # can be much larger; matters a lot once you're predicting ~4000 slices
}

CONFIG["model_dir"].mkdir(parents=True, exist_ok=True)
(CONFIG["output_dir"] / "masks_unet").mkdir(exist_ok=True)
CONFIG


## 2. Load the manifest, split labelled vs. unlabelled

`corrected == True` rows are your hand-verified ground-truth slices
(training data). Everything else among the 50 samples is what sections 8-9
predict on — if every sample is already corrected, `unlabelled` will just be
empty and those sections become a no-op; skip straight to section 11 for
full-stack segmentation.


In [ ]:
manifest_path = CONFIG["output_dir"] / "sample_manifest.csv"
manifest = pd.read_csv(manifest_path)

def sample_filename(row):
    return f"sample_{row.sample_id:03d}_{Path(row.filename).stem}.tif"

labelled = manifest.loc[manifest.corrected].reset_index(drop=True)
unlabelled = manifest.loc[~manifest.corrected].reset_index(drop=True)

print(f"{len(labelled)} ground-truth (corrected) slices, {len(unlabelled)} remaining to segment")
assert len(labelled) >= 4, "Need at least a handful of corrected slices before training — check output_dir."

train_rows, val_rows = train_test_split(
    labelled, test_size=CONFIG["val_fraction"], random_state=None  # reshuffled every run, not reproducible
)
print(f"{len(train_rows)} training slices, {len(val_rows)} validation slices")
print(f"Validation sample_ids: {sorted(val_rows['sample_id'].tolist())}")

## 3. Intensity normalization

XCT slices are raw attenuation values, not 0-255 images — rescale using
percentiles computed from the training slices only (avoids leaking
validation/inference statistics into the normalization), then replicate to
3 channels since the pretrained encoder expects RGB-shaped input.


In [ ]:
def load_image(fname):
    return tifffile.imread(CONFIG["output_dir"] / "images" / fname).astype(np.float32)

def load_mask(fname):
    return tifffile.imread(CONFIG["output_dir"] / "masks_corrected" / fname)

_sample_imgs = [load_image(sample_filename(row)) for _, row in train_rows.iterrows()]
_all_vals = np.concatenate([im.ravel() for im in _sample_imgs])
P_LOW, P_HIGH = np.percentile(_all_vals, [1, 99])
print(f"Normalizing with 1st/99th percentiles from training slices: {P_LOW:.1f} / {P_HIGH:.1f}")
del _sample_imgs, _all_vals

def normalize_to_uint8(img):
    clipped = np.clip(img, P_LOW, P_HIGH)
    scaled = (clipped - P_LOW) / (P_HIGH - P_LOW + 1e-8)
    return (scaled * 255).astype(np.uint8)

def to_rgb(img_uint8):
    return np.stack([img_uint8] * 3, axis=-1)

def one_hot(mask):
    return np.stack([(mask == c) for c in range(CONFIG["n_classes"])], axis=-1).astype(np.float32)

preprocess_input = sm.get_preprocessing(CONFIG["backbone"])


## 4. Training data: random augmented patches

`PatchGenerator` preloads all training images/masks into memory once (small
enough at typical ground-truth counts for this project), then serves random
augmented crops each batch.
Augmentation matters a lot here given how few labelled slices there are —
flips/rotations/brightness jitter effectively multiply the training set.


In [ ]:
train_transform = A.Compose([
    A.RandomCrop(CONFIG["patch_size"], CONFIG["patch_size"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
])

class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, rows, batch_size, steps, patch_size):
        self.rows = rows.reset_index(drop=True)
        self.batch_size = batch_size
        self.steps = steps
        self.patch_size = patch_size
        self.rng = np.random.default_rng(CONFIG["random_seed"])

        self.images, self.masks = {}, {}
        for _, row in self.rows.iterrows():
            fname = sample_filename(row)
            self.images[row.sample_id] = normalize_to_uint8(load_image(fname))
            self.masks[row.sample_id] = load_mask(fname)

    def __len__(self):
        return self.steps

    def __getitem__(self, idx):
        batch_imgs, batch_masks = [], []
        for _ in range(self.batch_size):
            sid = self.rng.choice(self.rows["sample_id"].values)
            img_rgb = to_rgb(self.images[sid])
            mask = self.masks[sid]

            aug = train_transform(image=img_rgb, mask=mask)
            patch_img = preprocess_input(aug["image"].astype(np.float32))
            patch_mask = one_hot(aug["mask"])

            batch_imgs.append(patch_img)
            batch_masks.append(patch_mask)

        return np.stack(batch_imgs), np.stack(batch_masks)

train_steps = max(1, CONFIG["patches_per_epoch"] // CONFIG["batch_size"])
train_generator = PatchGenerator(train_rows, CONFIG["batch_size"], train_steps, CONFIG["patch_size"])
print(f"{train_steps} steps/epoch, {CONFIG['batch_size']} patches/step")


## 5. Validation data: fixed grid tiles

Unlike training, validation uses the same fixed set of tiles every epoch
(a small grid per slice, no randomness) so `val_loss`/`val_iou_score` are
directly comparable across epochs instead of jumping around from resampling.


In [ ]:
def extract_grid_tiles(img_rgb, mask, patch_size, n_tiles):
    H, W = mask.shape
    n_side = int(np.ceil(np.sqrt(n_tiles)))
    ys = np.linspace(0, max(H - patch_size, 0), n_side).astype(int)
    xs = np.linspace(0, max(W - patch_size, 0), n_side).astype(int)
    tiles = []
    for y in ys:
        for x in xs:
            if len(tiles) >= n_tiles:
                break
            tiles.append((img_rgb[y:y + patch_size, x:x + patch_size],
                          mask[y:y + patch_size, x:x + patch_size]))
    return tiles

X_val, Y_val = [], []
for _, row in val_rows.iterrows():
    fname = sample_filename(row)
    img_rgb = to_rgb(normalize_to_uint8(load_image(fname)))
    mask = load_mask(fname)
    for patch_img, patch_mask in extract_grid_tiles(img_rgb, mask, CONFIG["patch_size"], CONFIG["val_tiles_per_image"]):
        X_val.append(preprocess_input(patch_img.astype(np.float32)))
        Y_val.append(one_hot(patch_mask))

X_val = np.stack(X_val)
Y_val = np.stack(Y_val)
print(f"Validation set: {X_val.shape[0]} tiles from {len(val_rows)} slice(s)")


## 6. Build and compile the U-Net

Dice + categorical focal loss handles the class imbalance reasonably well
(voidage dominates the pixel count; membrane and polymer are minority
classes) without needing manually-tuned class weights.


In [ ]:
model = sm.Unet(
    CONFIG["backbone"],
    classes=CONFIG["n_classes"],
    activation="softmax",
    encoder_weights="imagenet",
    input_shape=(CONFIG["patch_size"], CONFIG["patch_size"], 3),
)

total_loss = sm.losses.DiceLoss(class_weights=np.array([1.0,3.0,1.0])) + sm.losses.CategoricalFocalLoss()

model.compile(
    optimizer=tf.keras.optimizers.Adam(CONFIG["learning_rate"]),
    loss=total_loss,
    metrics=[sm.metrics.IOUScore(threshold=0.5), sm.metrics.FScore(threshold=0.5)],
)

model.summary()


## 7. Train

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(CONFIG["model_dir"] / "best_model.h5"),
        save_best_only=True, monitor="val_iou_score", mode="max",
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_iou_score", mode="max", factor=0.5, patience=5),
    tf.keras.callbacks.EarlyStopping(monitor="val_iou_score", mode="max", patience=12, restore_best_weights=True),
]

history = model.fit(
    train_generator,
    validation_data=(X_val, Y_val),
    epochs=CONFIG["epochs"],
    callbacks=callbacks,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history.history["iou_score"], label="train")
axes[1].plot(history.history["val_iou_score"], label="val")
axes[1].set_title("IoU score"); axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
model.save(str(CONFIG["model_dir"] / "final_model.h5"))
print("Saved model ->", CONFIG["model_dir"] / "final_model.h5")


### 7b. Reload a previously-trained model instead of retraining

If you're resuming (e.g. after a kernel restart mid-way through section 11)
and already have a saved model, **run this instead of section 7** — no
need to retrain. Rebuilds the architecture fresh (`encoder_weights=None`,
since real weights get loaded right after and there's no point downloading
ImageNet weights just to overwrite them) then loads the saved weights
directly via `load_weights()` rather than `load_model()` — this sidesteps
needing to pass the exact custom loss/metric objects back in as
`custom_objects`, which `load_model()` would otherwise require.


In [ ]:
model = sm.Unet(
    CONFIG["backbone"],
    classes=CONFIG["n_classes"],
    activation="softmax",
    encoder_weights=None,
    input_shape=(CONFIG["patch_size"], CONFIG["patch_size"], 3),
)
model.load_weights(str(CONFIG["model_dir"] / "final_model.h5"))  # or best_model.h5
print("Loaded weights ->", CONFIG["model_dir"] / "final_model.h5")


## 8. Tiled inference on full-resolution slices

Slides the trained patch-sized model over a full slice with overlap,
averaging overlapping softmax predictions before taking the final class —
this avoids the harsh tile-boundary artifacts a non-overlapping grid would
produce.

**Speed note:** `model.predict()` carries real per-call overhead (input
validation, progress-bar setup, a data-adapter pass) meant for iterating
large datasets, not for being called thousands of times in a tight loop
across ~4000 slices. `_predict_step` below is a JIT-compiled (`XLA`,
`jit_compile=True`) direct call instead — same math, no Keras dataset
machinery in the way. It's scoped to inference only (not used during
training above), so it doesn't touch anything that already worked. Batches
are padded to a fixed size so the compiled graph is only traced once instead
of re-tracing on every differently-sized final batch.


In [ ]:
@tf.function(jit_compile=True)
def _predict_step(x):
    return model(x, training=False)

def _predict_batches(batch, batch_size):
    preds = []
    for i in range(0, len(batch), batch_size):
        chunk = batch[i:i + batch_size]
        pad_n = batch_size - len(chunk)
        if pad_n > 0:
            pad = np.zeros((pad_n, *chunk.shape[1:]), dtype=chunk.dtype)
            chunk = np.concatenate([chunk, pad], axis=0)
        pred = _predict_step(tf.constant(chunk)).numpy()
        if pad_n > 0:
            pred = pred[:-pad_n]
        preds.append(pred)
    return np.concatenate(preds, axis=0)

def predict_full_image(img_raw, patch_size=None, stride=None):
    """Always predicts using the global `model` (via _predict_step, which is
    compiled against it) — no model argument, so there's no risk of silently
    predicting with a stale/different model object than intended."""
    patch_size = patch_size or CONFIG["patch_size"]
    stride = stride or max(1, int(patch_size * CONFIG["inference_stride_fraction"]))

    img_rgb = to_rgb(normalize_to_uint8(img_raw))
    H, W = img_raw.shape

    pad_h = (patch_size - H % stride) % stride
    pad_w = (patch_size - W % stride) % stride
    img_padded = np.pad(img_rgb, ((0, pad_h + patch_size), (0, pad_w + patch_size), (0, 0)), mode="reflect")
    Hp, Wp = img_padded.shape[:2]

    ys = list(range(0, Hp - patch_size + 1, stride))
    xs = list(range(0, Wp - patch_size + 1, stride))

    batch, positions = [], []
    for y in ys:
        for x in xs:
            tile = img_padded[y:y + patch_size, x:x + patch_size]
            batch.append(preprocess_input(tile.astype(np.float32)))
            positions.append((y, x))
    batch = np.stack(batch)

    preds = _predict_batches(batch, CONFIG["inference_batch_size"])

    prob_sum = np.zeros((Hp, Wp, CONFIG["n_classes"]), dtype=np.float32)
    weight = np.zeros((Hp, Wp), dtype=np.float32)
    for (y, x), pred in zip(positions, preds):
        prob_sum[y:y + patch_size, x:x + patch_size] += pred
        weight[y:y + patch_size, x:x + patch_size] += 1

    prob_avg = prob_sum[:H, :W] / np.maximum(weight[:H, :W, None], 1e-8)
    return np.argmax(prob_avg, axis=-1).astype(np.uint8)

def overlay(img, labels):
    img_norm = (img - img.min()) / max(1, (img.max() - img.min()))
    return label2rgb(labels.astype(np.int32), image=img_norm, bg_label=-1, alpha=0.35)


In [ ]:
unet_output_dir = CONFIG["output_dir"] / "masks_unet"

if len(unlabelled) == 0:
    print("Every sample is already corrected — nothing to predict here, skip to section 11.")
else:
    for _, row in tqdm(list(unlabelled.iterrows()), desc="U-Net inference"):
        fname = sample_filename(row)
        img_raw = load_image(fname)
        pred_mask = predict_full_image(img_raw)
        tifffile.imwrite(unet_output_dir / fname, pred_mask)

    print(f"Saved {len(unlabelled)} predicted masks -> {unet_output_dir}")


## 9. QC: spot-check predictions on the unlabelled slices

In [ ]:
if len(unlabelled) == 0:
    print("Every sample is already corrected — nothing to preview here, skip to section 11.")
else:
    n_preview = 6
    preview_rows = unlabelled.sample(n=min(n_preview, len(unlabelled)), random_state=CONFIG["random_seed"])

    fig, axes = plt.subplots(2, len(preview_rows), figsize=(3 * len(preview_rows), 6))
    for col, (_, row) in enumerate(preview_rows.iterrows()):
        fname = sample_filename(row)
        img = load_image(fname)
        pred = tifffile.imread(unet_output_dir / fname)

        axes[0, col].imshow(img, cmap="gray")
        axes[0, col].set_title(f"sample {row.sample_id}", fontsize=9)
        axes[0, col].axis("off")

        axes[1, col].imshow(overlay(img, pred))
        axes[1, col].axis("off")

    axes[0, 0].set_ylabel("raw")
    axes[1, 0].set_ylabel("U-Net prediction")
    plt.tight_layout()
    plt.show()


## 10. Segment the full ~4000-slice stack

Runs every page of the original stack through the model (using
`predict_full_image()` from section 8), except for the 50 sampled slices
where you have real hand-corrected ground truth — those are copied directly
rather than re-predicted, since your correction is more trustworthy than the
model's guess. Everything else gets a model prediction.

**This will take a long time** — likely hours, extrapolate roughly from
however long section 8's inference took per slice, times ~4000. A few things
make it practical to just let run:

- **Resumable.** Each slice writes to its own file (`slice_00000.tif`,
  `slice_00001.tif`, ...) in `full_stack_masks/`, written via a temp file +
  atomic rename so a slice is never left half-written. If the kernel dies or
  you stop it partway, just re-run this cell — it skips every slice that
  already has a completed output file rather than starting over.
- **Disk space**: ~4-4.5MB per slice as uint8 → roughly 16-18GB for the
  full stack. Worth checking free space before starting, especially if
  `output_dir` is on external/USB storage.
- `inference_batch_size` (section 1) controls GPU throughput per slice —
  raise it if you have GPU memory to spare, lower it if you hit
  out-of-memory errors partway through.


In [ ]:
tif_file = tifffile.TiffFile(CONFIG["stack_path"])
n_total = len(tif_file.pages)
print(f"Found {n_total} pages in {CONFIG['stack_path']}")

FULL_MASKS_DIR = CONFIG["output_dir"] / "full_stack_masks"
FULL_MASKS_DIR.mkdir(exist_ok=True)

# use your real corrected ground truth directly wherever it exists, rather
# than re-predicting via the model
gt_lookup = {}
for _, row in manifest.loc[manifest.corrected].iterrows():
    gt_lookup[row.slice_index] = CONFIG["output_dir"] / "masks_corrected" / sample_filename(row)

print(f"{len(gt_lookup)} slice(s) will use ground truth directly; "
      f"{n_total - len(gt_lookup)} will be predicted by the model.")


In [ ]:
FORCE_RESEGMENT = False  # set True to overwrite every slice with the current model, e.g. after retraining

n_skipped, n_done = 0, 0
for idx in tqdm(range(n_total), desc="Segmenting full stack"):
    out_path = FULL_MASKS_DIR / f"slice_{idx:05d}.tif"
    if out_path.exists() and not FORCE_RESEGMENT:
        n_skipped += 1
        continue

    if idx in gt_lookup:
        pred_mask = tifffile.imread(gt_lookup[idx])
    else:
        img_raw = tif_file.pages[idx].asarray().astype(np.float32)
        pred_mask = predict_full_image(img_raw)

    tmp_path = out_path.with_suffix(".tmp.tif")
    tifffile.imwrite(tmp_path, pred_mask)
    tmp_path.replace(out_path)  # atomic rename — never leaves a half-written file at out_path
    n_done += 1

print(f"Done. {n_done} slice(s) segmented this run, {n_skipped} already existed and were skipped.")


### QC: spot-check a few slices from across the full stack, not just the sampled ones

In [ ]:
preview_indices = sorted(np.random.default_rng(CONFIG['random_seed']).choice(n_total, size=6, replace=False).tolist())

fig, axes = plt.subplots(2, len(preview_indices), figsize=(3 * len(preview_indices), 6))
for col, idx in enumerate(preview_indices):
    img = tif_file.pages[idx].asarray().astype(np.float32)
    pred = tifffile.imread(FULL_MASKS_DIR / f"slice_{idx:05d}.tif")

    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(f"slice {idx}", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(overlay(img, pred))
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("raw")
axes[1, 0].set_ylabel("prediction")
plt.tight_layout()
plt.show()


In [ ]:
tif_file.close()


## 12. Next steps

- If predictions look weak in specific regions, hand-correcting a few more
  slices (using the annotation notebook — you can add more samples to
  `sample_manifest.csv` manually, or increase `n_samples` there and re-run
  its sampling cell) and re-running this notebook is usually more effective
  than tuning hyperparameters at this scale of ground truth.
- `full_stack_masks/` now holds one label mask per original slice
  (`0 = voidage`, `1 = membrane`, `2 = polymer_cartridge`) — this is the
  actual deliverable for downstream analysis/mechanistic modelling.
- `CONFIG["model_dir"] / "best_model.h5"` holds the best checkpoint by
  validation IoU if you want to reload it later without retraining:
  `model = tf.keras.models.load_model(path, custom_objects={...})` — needs
  the same custom loss/metric objects from `segmentation_models` passed in
  via `custom_objects`, or reconstruct the architecture and call
  `model.load_weights(path)` instead, which sidesteps that entirely.
